In [2]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer
import jinja2

In [3]:
lib = pd.read_csv('LIB.csv')
corpus = pd.read_csv('CORPUS.csv')
vocab = pd.read_csv('VOCAB.csv')

In [4]:
OHCO = ['album_id', 'song_id', 'token_id']
bags = dict(
    ALBUM = OHCO[:2],
    SONG = OHCO[:1]
)
token = corpus.set_index(OHCO).dropna()

### BOW

In [5]:
bag = 'ALBUM'

bags[bag]+['term_str']
bow = token.groupby(bags[bag]+['term_str']).term_str.count().to_frame('n') 
bow.head()

n
album_id song_id term_str   
0        115     a         4
                 aalright  2
                 aint      1
                 all       3
                 alright   2

### DTM

In [6]:
dtm = bow.n.unstack(fill_value=0)
dtm.head()

term_str          10  1159  12  123  17  23  3  300  35  4am  ...  za  zbog  \
album_id song_id                                              ...             
0        115       0     0   0    0   0   0  0    0   0    0  ...   0     0   
         116       0     0   0    0   0   0  0    0   0    0  ...   0     0   
         117       0     0   0    0   0   0  0    0   0    0  ...   0     0   
         118       0     0   0    0   0   0  0    0   0    0  ...   0     0   
         119       0     0   0    0   0   0  0    0   0    0  ...   0     0   

term_str          zedd  zip  zippy  zlo  zombi  zombie  zombieboy  zone  
album_id song_id                                                         
0        115         0    0      0    0      0       0          0     0  
         116         0    0      0    0      0       0          0     0  
         117         0    0      0    0      0       0          0     0  
         118         0    0      0    0      0       0          0     0  
         119         0    0      0    0      0       0          0     0  

[5 rows x 3048 columns]

### TFIDF

In [7]:
tf_method = 'double_norm'
tf_norm_k = .5
idf_method = 'standard'
gradient_cmap = 'YlGnBu'

In [8]:
if tf_method == 'sum':
    TF = dtm.T / dtm.T.sum()

elif tf_method == 'smooth':
    TF = (dtm.T / dtm.T.sum()) + 1

elif tf_method == 'max':
    TF = dtm.T / dtm.T.max()
    
elif tf_method == 'log':
    TF = np.log2(1 + dtm.T)
    
elif tf_method == 'raw':
    TF = dtm.T
    
elif tf_method == 'double_norm':
    TF = ((dtm.T + .5) / (dtm.T.max() + .5)) + .5
    
elif tf_method == 'binary':
    TF = dtm.T.astype('bool').astype('int')
    
tf = TF.T

In [9]:
df = dtm.astype('bool').sum() 
n = dtm.shape[0]
idf = np.log2(n / df)

tfidf = tf * idf
tfidf.head()

term_str                10      1159        12       123        17        23  \
album_id song_id                                                               
0        115      3.630207  3.630207  2.810862  3.630207  3.630207  3.630207   
         116      3.593800  3.593800  2.782672  3.593800  3.593800  3.593800   
         117      3.691245  3.691245  2.858123  3.691245  3.691245  3.691245   
         118      3.619220  3.619220  2.802355  3.619220  3.619220  3.619220   
         119      3.583579  3.583579  2.774758  3.583579  3.583579  3.583579   

term_str                 3       300        35       4am  ...        za  \
album_id song_id                                          ...             
0        115      3.630207  3.113258  3.630207  3.630207  ...  3.630207   
         116      3.593800  3.082035  3.593800  3.593800  ...  3.593800   
         117      3.691245  3.165604  3.691245  3.691245  ...  3.691245   
         118      3.619220  3.103836  3.619220  3.619220  ...  3.619220   
         119      3.583579  3.073270  3.583579  3.583579  ...  3.583579   

term_str              zbog      zedd       zip     zippy       zlo     zombi  \
album_id song_id                                                               
0        115      3.630207  3.630207  3.630207  3.630207  3.630207  3.630207   
         116      3.593800  3.593800  3.593800  3.593800  3.593800  3.593800   
         117      3.691245  3.691245  3.691245  3.691245  3.691245  3.691245   
         118      3.619220  3.619220  3.619220  3.619220  3.619220  3.619220   
         119      3.583579  3.583579  3.583579  3.583579  3.583579  3.583579   

term_str            zombie  zombieboy      zone  
album_id song_id                                 
0        115      3.630207   3.630207  3.113258  
         116      3.593800   3.593800  3.082035  
         117      3.691245   3.691245  3.165604  
         118      3.619220   3.619220  3.103836  
         119      3.583579   3.583579  3.073270  

[5 rows x 3048 columns]

In [10]:
vocab['df'] = df
vocab['idf'] = idf
vocab['tfidf_mean'] = tfidf.mean()
bow['tf'] = tf.stack()
bow['tfidf'] = tfidf.stack()

### L2 TFIDF

In [11]:
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA, TruncatedSVD as SVD

tfidfL2 = pd.DataFrame(normalize(tfidf, norm='l2', axis=1), index=tfidf.index, columns=tfidf.columns)
tfidfL2.head()

term_str                10      1159        12       123        17        23  \
album_id song_id                                                               
0        115      0.020500  0.020500  0.015873  0.020500  0.020500  0.020500   
         116      0.020516  0.020516  0.015885  0.020516  0.020516  0.020516   
         117      0.020514  0.020514  0.015884  0.020514  0.020514  0.020514   
         118      0.020495  0.020495  0.015870  0.020495  0.020495  0.020495   
         119      0.020538  0.020538  0.015902  0.020538  0.020538  0.020538   

term_str                 3       300        35       4am  ...        za  \
album_id song_id                                          ...             
0        115      0.020500  0.017581  0.020500  0.020500  ...  0.020500   
         116      0.020516  0.017594  0.020516  0.020516  ...  0.020516   
         117      0.020514  0.017593  0.020514  0.020514  ...  0.020514   
         118      0.020495  0.017577  0.020495  0.020495  ...  0.020495   
         119      0.020538  0.017613  0.020538  0.020538  ...  0.020538   

term_str              zbog      zedd       zip     zippy       zlo     zombi  \
album_id song_id                                                               
0        115      0.020500  0.020500  0.020500  0.020500  0.020500  0.020500   
         116      0.020516  0.020516  0.020516  0.020516  0.020516  0.020516   
         117      0.020514  0.020514  0.020514  0.020514  0.020514  0.020514   
         118      0.020495  0.020495  0.020495  0.020495  0.020495  0.020495   
         119      0.020538  0.020538  0.020538  0.020538  0.020538  0.020538   

term_str            zombie  zombieboy      zone  
album_id song_id                                 
0        115      0.020500   0.020500  0.017581  
         116      0.020516   0.020516  0.017594  
         117      0.020514   0.020514  0.017593  
         118      0.020495   0.020495  0.017577  
         119      0.020538   0.020538  0.017613  

[5 rows x 3048 columns]

In [12]:
tfidf.to_csv('tfidf.csv')
tfidfL2.to_csv('tfidfL2.csv')